In [3]:
import torch
import torch.nn as nn

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, stride, 1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, 3, 1, 1),
            nn.BatchNorm2d(out_channels)
        )
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        return torch.relu(self.conv(x) + self.shortcut(x))

class ASLResNet(nn.Module):
    def __init__(self, num_classes=25): # Labels go up to 24
        super().__init__()
        self.in_layer = nn.Sequential(
            nn.Conv2d(1, 64, 3, 1, 1),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )
        self.res1 = ResidualBlock(64, 64)
        self.res2 = ResidualBlock(64, 128, stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.in_layer(x)
        x = self.res1(x)
        x = self.res2(x)
        x = self.avgpool(x)
        return self.fc(x.view(x.size(0), -1))

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

class ASLDataset(Dataset):
    def __init__(self, csv_file, transform=None):
        data = pd.read_csv(csv_file)
        self.labels = torch.tensor(data.iloc[:, 0].values, dtype=torch.long)
        self.images = torch.tensor(data.iloc[:, 1:].values, dtype=torch.float32)
        self.images = self.images.view(-1, 1, 28, 28) / 255.0  # Reshape & Normalize
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        image = self.images[idx]
        if self.transform:
            image = self.transform(image)
        return image, self.labels[idx]

# Initialize Loaders
train_dataset = ASLDataset('/Users/ishaanagarwal/Desktop/Cornell/STSCI 5720 Applied Neural Networks/Final Project/Data/sign_mnist_train.csv')
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_dataset = ASLDataset('/Users/ishaanagarwal/Desktop/Cornell/STSCI 5720 Applied Neural Networks/Final Project/Data/sign_mnist_test.csv')
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=True)

In [5]:
import torch

# Check if MPS is available (for M-series Mac or newer Intel Macs with AMD GPUs)
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS (Metal Performance Shaders) for acceleration.")
else:
    device = torch.device("cpu")
    print("MPS not available, falling back to CPU.")

# When moving your model and data:
model = ASLResNet(num_classes=26).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

def train_model(model, loader, epochs=5):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        correct = 0
        total = 0
        
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
        print(f"Epoch {epoch+1}: Loss: {running_loss/len(loader):.4f} | Acc: {100.*correct/total:.2f}%")

train_model(model, train_loader)

Using MPS (Metal Performance Shaders) for acceleration.
Epoch 1: Loss: 1.0948 | Acc: 74.50%
Epoch 2: Loss: 0.0713 | Acc: 99.73%
Epoch 3: Loss: 0.0106 | Acc: 100.00%
Epoch 4: Loss: 0.0041 | Acc: 100.00%
Epoch 5: Loss: 0.0287 | Acc: 99.58%


In [6]:
import torch.nn.functional as F

class SimpleViT(nn.Module):
    def __init__(self, image_size=28, patch_size=7, num_classes=26, dim=64, depth=4, heads=4, mlp_dim=128):
        super().__init__()
        num_patches = (image_size // patch_size) ** 2
        patch_dim = patch_size ** 2 # Because it's grayscale (1 channel)
        
        # 1. Patch Embedding
        self.to_patch_embedding = nn.Linear(patch_dim, dim)
        self.pos_embedding = nn.Parameter(torch.randn(1, num_patches + 1, dim))
        self.cls_token = nn.Parameter(torch.randn(1, 1, dim))
        
        # 2. Transformer Blocks
        encoder_layer = nn.TransformerEncoderLayer(d_model=dim, nhead=heads, dim_feedforward=mlp_dim, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)
        
        # 3. MLP Head
        self.mlp_head = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, num_classes)
        )

    def forward(self, img):
        # Flatten image into patches: [batch, 1, 28, 28] -> [batch, 16, 49]
        p = 7
        x = img.unfold(2, p, p).unfold(3, p, p)
        x = x.contiguous().view(img.size(0), -1, p*p)
        
        # Embed patches and add CLS token
        x = self.to_patch_embedding(x)
        b, n, _ = x.shape
        cls_tokens = self.cls_token.expand(b, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        x += self.pos_embedding
        
        # Transformer
        x = self.transformer(x)
        
        # Classify based on CLS token
        return self.mlp_head(x[:, 0])

# Initialize ViT
vit_model = SimpleViT().to(device)

In [7]:
import torch.optim as optim

# 1. Setup Model, Optimizer, and Loss
# We use 26 classes to account for the label range 0-25
model_vit = SimpleViT(image_size=28, patch_size=7, num_classes=26).to(device)
optimizer = optim.Adam(model_vit.parameters(), lr=0.0005) # ViTs are sensitive to LR
criterion = nn.CrossEntropyLoss()

def train_and_evaluate_vit(model, train_loader, epochs=1):
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        correct = 0
        total = 0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            # Backward and optimize
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
        acc = 100 * correct / total
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {train_loss/len(train_loader):.4f}, Accuracy: {acc:.2f}%')

# Run the training
train_and_evaluate_vit(model_vit, train_loader)

Epoch [1/1], Loss: 2.5995, Accuracy: 20.76%


In [8]:
def test_accuracy(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    print(f'Final Accuracy on Test Set: {100 * correct / total:.2f}%')

# Assuming you've created a test_loader for sign_mnist_test.csv
test_accuracy(model_vit, test_loader)
test_accuracy(model, test_loader)

Final Accuracy on Test Set: 46.49%
Final Accuracy on Test Set: 99.83%


In [ ]:
import cv2
import numpy as np

# Initialize webcam
cap = cv2.VideoCapture(0)
alphabet = "ABCDEFGHIKLMNOPQRSTUVWXY"

while True:
    ret, frame = cap.read()
    if not ret: break

    # 1. Mirror the frame (makes it easier to position your hand)
    frame = cv2.flip(frame, 1)
    
    # 2. Get frame dimensions
    height, width, _ = frame.shape
    
    # 3. Calculate Center Coordinates for a 300x300 box
    size = 300
    x1 = int((width - size) / 2)
    y1 = int((height - size) / 2)
    x2 = x1 + size
    y2 = y1 + size

    # 4. Define and extract the ROI
    cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 0), 2) # Blue box
    roi = frame[y1:y2, x1:x2]

    # 5. Pre-process for the model (Grayscale -> 28x28)
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(gray, (28, 28))
    
    # Prep tensor
    img_tensor = torch.FloatTensor(resized).unsqueeze(0).unsqueeze(0) / 255.0
    img_tensor = img_tensor.to(device)

    # 6. Predict
    model.eval()
    with torch.no_grad():
        output = model(img_tensor)
        _, pred = torch.max(output, 1)
        letter = alphabet[pred.item()] if pred.item() < len(alphabet) else "?"

    # 7. Display Prediction centered above the box
    cv2.putText(frame, f"Letter: {letter}", (x1, y1 - 10), 
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 0, 0), 2)
    
    cv2.imshow('Centered ASL Recognition', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

: 